# ORC Basics — 03: ORC Internals

ORC's internal structure explains why it performs well for analytics.
Understanding stripes, row indexes, and statistics helps you tune writes.

Topics: file footer, stripe footer, row index, column encodings, ACID ORC.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/orc_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('orc-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — ORC File Structure

3-level layout: file → stripe → row index.

In [ ]:

print("""
ORC File Layout:
  ┌──────────────────────────────────────────┐
  │  Postscript (last bytes: version, magic) │
  │  File Footer                             │
  │    - List of stripes (offset + length)   │
  │    - Column statistics (entire file)     │
  │    - Schema                              │
  ├──────────────────────────────────────────┤
  │  Stripe N (default 64 MB)                │
  │    Row Index (every 10,000 rows)         │
  │      - Column stats per 10K row group    │
  │    Row Data                              │
  │      - Column streams (encoded data)     │
  │    Stripe Footer                         │
  │      - Stream locations + encodings      │
  ├──────────────────────────────────────────┤
  │  Stripe N-1 ...                          │
  └──────────────────────────────────────────┘

3 levels of skipping:
  1. File footer: skip entire file if stats don't match
  2. Stripe footer: skip entire stripe (64 MB)
  3. Row index: skip 10,000-row blocks within stripe
""")


## Step 2 — Column Encodings

ORC uses encodings similar to Parquet but with ORC-specific types.

In [ ]:

print("""
ORC column encodings:
  DIRECT        - raw bytes (strings with high cardinality)
  DICTIONARY    - dictionary + integer indices (low cardinality strings)
  DIRECT_V2     - improved direct with RLE v2 (integers)
  DICTIONARY_V2 - improved dictionary encoding

Integer encoding (RLE v2 = Run Length Encoding):
  4 sub-encodings: SHORT_REPEAT, DIRECT, PATCHED_BASE, DELTA
  DELTA: great for sequential IDs (order_id, timestamp)
  SHORT_REPEAT: great for repeated values (status codes)

Boolean columns: packed 8 per byte with RLE
Timestamp: seconds + nanoseconds (higher precision than Parquet)
""")

# Verify in explain — look for OrcScan
ORC_PATH = f"{DATA_DIR}/orders.orc"
import pathlib, glob as glib
if not pathlib.Path(ORC_PATH).exists():
    import random, datetime
    random.seed(42)
    N=50000; base=datetime.date(2023,1,1)
    df=spark.range(N).select(F.col("id").alias("order_id"),
        F.element_at(F.array([F.lit(c) for c in ["Electronics","Books","Food"]]),(F.col("id")%3+1).cast("int")).alias("category"),
        F.round(F.rand(42)*500,2).alias("revenue"),
        (F.date_add(F.lit("2023-01-01"),(F.col("id")%365).cast("int"))).cast("date").alias("order_date"))
    df.write.mode("overwrite").orc(ORC_PATH)

print("\nORC scan plan with predicate pushdown:")
spark.read.orc(ORC_PATH).filter(F.col("category")=="Electronics").explain(mode="simple")


## Step 3 — Statistics: min/max/sum/count

ORC stores richer statistics than Parquet (includes sum for numeric columns).

In [ ]:

ORC_PATH = f"{DATA_DIR}/orders.orc"
try:
    import pyarrow.orc as po
    orc_file = glib.glob(f"{ORC_PATH}/*.orc")[0]
    f = po.ORCFile(orc_file)
    print(f"File: {orc_file.split('/')[-1]}")
    print(f"  Rows   : {f.nrows:,}")
    print(f"  Stripes: {f.nstripes}")
    print()
    schema = f.schema
    print("Column types from ORC schema:")
    for i, name in enumerate(schema.names[:8]):
        print(f"  [{i}] {name}: {schema.field(name).type}")
except ImportError:
    print("pyarrow.orc not available")
    print("ORC statistics accessible via Spark explain():")
    spark.read.orc(ORC_PATH) \
         .filter(F.col("revenue").between(100, 500)) \
         .explain(mode="formatted")


## Step 4 — ORC ACID (Hive ACID Format)

Legacy Hive ACID ORC uses delta directories for updates/deletes.

In [ ]:

print("""
Hive ACID ORC format (you may encounter this when reading Hive tables):

  table_dir/
    base_0000001/        ← base files (full data snapshot)
      bucket_00000.orc
    delta_0000002_0000002/  ← INSERT delta
      bucket_00000.orc
    delete_delta_0000003_0000003/  ← DELETE markers
      bucket_00000.orc

Reading Hive ACID ORC in Spark:
  spark.read.orc("/hive/warehouse/table/")
  # Spark handles base + delta merge automatically

Note: Modern tables should use Delta Lake or Iceberg instead of ACID ORC.
ACID ORC is a legacy format maintained for Hive compatibility.
""")


## Summary



In [ ]:

print("""
ORC internals summary:
  Unit of I/O   : Stripe (default 64 MB) — tune with orc.stripe.size
  Row skipping  : Row Index every 10,000 rows — tune with orc.row.index.stride
  Statistics    : min/max/sum/count per column per stripe (more than Parquet)
  Encodings     : DICTIONARY/DIRECT/DELTA/RLE_V2 per column
  Bloom filters : Optional per-column (equality pushdown)
  Compression   : per stripe (zlib/snappy/lz4/zstd/none)

Key difference from Parquet:
  ORC: 3-level index (file/stripe/row-index)
  Parquet: 2-level (file/row-group)
  → ORC can skip more precisely within large stripes
""")
